# Gold-layer branch performance

This notebook creates a Gold-layer table called:
"banking.gold.branch_performance".
The goal is to aggregate banking data at the branch level
for analytics, KPI reporting, and dashboard visualization.

Data is collected from Silver-layer tables:

- customers
- accounts
- transactions
- branches

Final metrics generated per branch:

- total customers
- total accounts
- total deposits
- total transactions
- total transaction amount

In [0]:
%sql
-- Create Gold Table

CREATE OR REPLACE TABLE banking.gold.branch_performance AS

-- Customer to Branch Mapping : Identify which customer belongs to which branch.
-- Source Table: banking.silver.customers

WITH customer_branch AS (
    SELECT 
        c.customer_id, c.branch_code
    FROM banking.silver.customers c
 ),

-- Account Aggregation per Customer 
-- Calculated Metrics: 
    -- total_accounts, total_balance(Sum of all account balances)
-- Source Table: banking.silver.accounts

account_agg AS (
    SELECT 
        a.customer_id,
        COUNT(a.account_id) AS total_accounts,
        SUM(a.balance) AS total_balance
    FROM banking.silver.accounts a
    GROUP BY a.customer_id
),

-- Transaction Aggregation per Customer
-- Calculated Metrics:
    -- total_transactions:
    -- total_transaction_amount (Sum of all transaction amounts)
-- Source Tables:
-- banking.silver.transactions
-- banking.silver.accounts

txn_agg AS ( 
    SELECT
        a.customer_id,
        COUNT(t.txn_id) AS total_transactions,
        SUM(t.amount) AS total_transaction_amount
    FROM banking.silver.transactions t
    JOIN banking.silver.accounts a
        ON t.account_id = a.account_id
    GROUP BY a.customer_id
)
-- Final Branch-Level Aggregation
-- This query combines all previous aggregations and generates branch-level KPIs
-- Final Metrics:
    -- total_customers
    -- total_accounts
    -- total_deposits
    -- total_transactions
    -- total_transaction_amount
-- Main Source: banking.silver.branches
-- LEFT JOIN is used to: keep all branches even if some have no data

SELECT
    b.branch_code,
    b.branch_name,
    COUNT(DISTINCT cb.customer_id) AS total_customers,
    SUM(a.total_accounts) AS total_accounts,
    SUM(a.total_balance) AS total_deposits,
    SUM(t.total_transactions) AS total_transactions,
    SUM(t.total_transaction_amount) AS total_transaction_amount

-- Join Branches with Customers
FROM banking.silver.branches b
LEFT JOIN customer_branch cb
    ON b.branch_code = cb.branch_code

-- Join Account Aggregation
LEFT JOIN account_agg a
    ON cb.customer_id = a.customer_id

-- Join Transaction Aggregation
LEFT JOIN txn_agg t
    ON cb.customer_id = t.customer_id
GROUP BY
b.branch_code,
b.branch_name



## Validate Gold Table Creation  
This Spark SQL query counts the number of rows created in the Gold table.

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.branch_performance
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))